# Exercise 2. LoRa for Low-Resource Languages
NLP for social good is not just about reducing harmful outputs; it is also about making AI accessible across languages, not only English. Low- and medium-resource languages, from Nigerian Pidgin to Danish, are often left behind. 

```{figure} ../figures/class8/neural-space-low-resource.png
---
name: neural-space-low-resource
width: 100%
---
Fig. borrowed from [NeuralSpace blogpost](https://medium.com/neuralspace/challenges-in-using-nlp-for-low-resource-languages-and-how-neuralspace-solves-them-54a01356a71b) by Felix Laumann
```

Fine-tuning LLMs can help, but it is costly. LoRA (Low-Rank Adaptation) offers a parameter-efficient alternative, reducing trainable parameters by up to 10,000 times. In other words, rather than training all 8 billion parameters of a model like [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B-Base), LoRA updates only a small fraction.

## 2.1 Intro to LoRa?
When we're doing LoRa, we are esentially training a LoRa adapter that could technically be placed on other models (if the base architecture matches):

```{figure} ../figures/class8/lora_adapter.png
---
name: lora_adapter
width: 100%
---
From HF's [smol course](https://huggingface.co/learn/smol-course/en/unit1/3a)
```

If you're interested in the math behind this (but in an intuitive way), I encourage you to read Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/i/138081202/a-brief-introduction-to-lora). You can also read the original paper by {cite:t}`hu_lora_2021`. 

## 2.2 Setup
For the code implementation, we'll use the [PEFT](https://huggingface.co/docs/peft/en/index) and [TRL](https://huggingface.co/docs/trl/en/index) library by Hugging Face
```bash
source .venv/bin/activate
pip install peft trl
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers datasets
```

Let's import:

In [231]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
from peft import LoraConfig, AutoPeftModelForCausalLM

from trl import SFTTrainer, SFTConfig

## 2.3 Load Model & Data
For today's exercise, we'll try to make a smaller version of `SmolLM2` good at English to Danish machine translation

:::{admonition} You can use LoRa for much more than Translation :)
:class: dropdown, tip
As a simple introduction to LoRA, we're doing machine translation, but you can use this approach for anything you'd like really - feel free to switch out the dataset for something you'd like. Or use this notebook as a inspiration for the exam :).

See also this tutorial for instruction-tuning a danish language model using QLoRA -> [Tutorial: Finetuning Language Models](https://www.foundationmodels.dk/blog/2024/02/02/tutorial-finetuning-language-models.html)
:::

We'll load a smaller version of `SmolLM2`:

In [232]:
model_id ="HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id)

We'll load the Danish-English translation set, but only a subset with `[:n]` for `n` rows:

In [233]:
n_rows = 2000
train_ds = load_dataset("Helsinki-NLP/opus-100", "da-en", split=f"train[:{n_rows}]")

Let's look at the only column, "translation" to see how it is structured: 

Let's print a few:

In [234]:
for translation in train_ds["translation"][:5]:
    print(f"EN: {translation['en']}")
    print(f"DA: {translation['da']}")
    print()

EN: For the EEA Joint Committee
DA: På Det Blandede EØS-Udvalgs vegne

EN: Metal containing by weight at least 99,9 % of lead, provided that the content by weight of any other element does not exceed the limit specified in the following table:
DA: metal, der indeholder mindst 99,9 vægtprocent bly, forudsat ingen anden bestanddel indgår i mængder, der overstiger de i nedenstående skema anførte grænseværdier:

EN: Think.
DA: Tænk.

EN: Beth...
DA: Beth...

EN: With the Human Hibernation Project, we will be able to save our best men... frozen in their prime, for use when they are needed most.
DA: Vort projekt "Menneskelig dvale" gør det muligt at holde vores bedste mænd nedfrosset i deres bedste tilstand, for at bruge dem efter behov.



## 1.3 Prompt Templating
We want a `prompt` column that inserts the English sentence as the `Source` and the Danish example as the `target`in this formatting by{cite:t}`alves_steering_2023`:
```{figure} ../figures/class8/prompt-template-alves.png
---
name: prompt-template-alves
width: 80%
---
Prompt template by {cite:t}`alves_steering_2023`
```
X should be "English" and Y should be "Danish" in our context.

### Your Turn: Formatting the Prompt
:::{admonition} HANDS-ON
:class: red
1. Create a function called `def format_prompt(example)`
    - It should process a single row `example` in our dataset
    - Use the prompt template above to create a `prompt` with English as the source & Danish as the target.
    - Return a dictionary entry `{"prompt": prompt}`

2. Test the function on a single example in `train_ds`, printing the prompt and completion"
:::

#### Solution

In [235]:
def preprocess_function(example):
    translation = example["translation"]
    return {"messages": [{"role": "user", "content": f"Translate to Danish: {translation['en']}"}, {"role": "assistant", "content": f"{translation['da']}"}]}

### Adding a Prompt Column 
We can now add the prompt column to our ds using our new `format_prompt` column:

In [236]:
formatted_train_ds = train_ds.map(preprocess_function, batched=False)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [237]:
formatted_train_ds

Dataset({
    features: ['translation', 'messages'],
    num_rows: 2000
})

In [238]:
formatted_train_ds = formatted_train_ds.remove_columns(["translation"])

## 1.4 LoRa Config & Training
We'll start by configuring LoRA: 

In [239]:
rank = 16
peft_config = LoraConfig(
    r=rank,
    lora_alpha=rank * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [240]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=f"lora-{model_id.split('/')[-1]}-da-en",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        packing=True, # can speed up training
        chat_template_path=model_id,
    ),
    train_dataset=formatted_train_ds,
    peft_config=peft_config
)
trainer.train()

Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-fla

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step,Training Loss
10,4.241400
20,4.260500
30,4.193000
40,4.108300
50,4.009600
60,4.082800


TrainOutput(global_step=60, training_loss=4.149267832438151, metrics={'train_runtime': 84.5669, 'train_samples_per_second': 1.419, 'train_steps_per_second': 0.709, 'total_flos': 77257304886528.0, 'train_loss': 4.149267832438151, 'epoch': 1.0})

In [243]:
adapter_path = f"lora-{model_id.split('/')[-1]}-da-en/checkpoint-60"

# load model with adapter
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
model = AutoPeftModelForCausalLM.from_pretrained(adapter_path, device_map="auto", torch_dtype=torch.float16)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use mps


In [249]:
prompt = "Translate to Danish: I love to drive my car."
format_prompt = pipe.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
outputs = pipe(format_prompt, max_new_tokens=50, return_full_text=False)
print(outputs[0]["generated_text"])

Hilper av möjligt med kunne til et bållbom årnekost.
